# Calculate Cell Density based on Annotated Slices

## Write localization annotations to ndpa

In [1]:
import numpy as np
from torch.utils.data import DataLoader
import pyvips
from pathlib import Path
import random
import matplotlib.pyplot as plt

random.seed(25)
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from skan import draw, Skeleton
import pyvips
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
import pandas as pd
from scripts.filters import *
from scripts.utils import *
import os
import xml.etree.ElementTree as ET
import xml.dom.minidom
import openslide


def write_ndpa(df, NDPI_PATH, slice=1, slide=None):
    filtered_df = df[(df["brain_slice"] == slice) & (df["slide"] == slide)].copy()

    filtered_df["label"] = ""
    filtered_df["color"] = "#FF0000"
    filtered_df["width"] = filtered_df["x2"] - filtered_df["x1"]
    filtered_df["height"] = filtered_df["y2"] - filtered_df["y1"]

    filtered_df = filtered_df.rename(columns={"x1": "x", "y1": "y"})

    BOUNDING_BOXES = filtered_df[
        ["label", "color", "x", "y", "width", "height"]
    ].to_dict(orient="records")

    def px_to_ndpa(px, py, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm):
        """Convert a single pixel coordinate (top-left origin) to NDPA nanometres."""
        ndpa_x = (px - img_w / 2) * mpp_x_nm + offset_x_nm
        ndpa_y = (py - img_h / 2) * mpp_y_nm + offset_y_nm
        return int(round(ndpa_x)), int(round(ndpa_y))

    def build_ndpa(boxes, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm):

        fragments = []

        for i, box in enumerate(boxes):
            label = box.get("label", f"bbox_{i + 1}")
            color = box.get("color", "#FF0000")

            x, y, w, h = box["x"], box["y"], box["width"], box["height"]

            cx_px = x + w / 2
            cy_px = y + h / 2
            cx_nm, cy_nm = px_to_ndpa(
                cx_px, cy_px, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm
            )

            corners_px = [
                (x, y),
                (x + w, y),
                (x + w, y + h),
                (x, y + h),
            ]
            corners_nm = [
                px_to_ndpa(
                    cx2, cy2, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm
                )
                for cx2, cy2 in corners_px
            ]

            state = ET.Element("ndpviewstate", id=str(i + 1))
            ET.SubElement(state, "title").text = label
            ET.SubElement(state, "details").text = ""
            ET.SubElement(state, "x").text = str(cx_nm)
            ET.SubElement(state, "y").text = str(cy_nm)
            ET.SubElement(state, "z").text = "0"
            ET.SubElement(state, "lens").text = "20.000000"

            ann = ET.SubElement(
                state, "annotation", type="freehand", displayname=label, color=color
            )
            ET.SubElement(ann, "measuretype").text = "0"
            ET.SubElement(ann, "closed").text = "1"
            ET.SubElement(ann, "specialtype").text = "rectangle"

            pl = ET.SubElement(ann, "pointlist")
            for nx, ny in corners_nm:
                pt = ET.SubElement(pl, "point")
                ET.SubElement(pt, "x").text = str(nx)
                ET.SubElement(pt, "y").text = str(ny)

            ET.SubElement(ann, "specialtype").text = "rectangle"

            fragments.append(ET.tostring(state, encoding="unicode"))

        return fragments

    def pretty_join(fragments):
        parts = []
        for frag in fragments:
            pretty = xml.dom.minidom.parseString(frag).toprettyxml(indent="  ")
            lines = pretty.splitlines()
            body = "\n".join(l for l in lines if not l.startswith("<?xml"))
            parts.append(body.strip())

        inner = "\n".join(parts)
        return (
            '<?xml version="1.0" encoding="utf-8" standalone="yes"?>\n'
            "<annotations>\n"
            f"{inner}\n"
            "</annotations>\n"
        )

    if not os.path.isfile(NDPI_PATH):
        raise FileNotFoundError(f"NDPI file not found: {NDPI_PATH}")

    print(f"Opening slide: {NDPI_PATH}")
    slide = openslide.OpenSlide(NDPI_PATH)
    props = slide.properties

    img_w, img_h = slide.dimensions
    print(f"  Dimensions: {img_w} x {img_h} px")

    mpp_x = float(props.get(openslide.PROPERTY_NAME_MPP_X, 0))
    mpp_y = float(props.get(openslide.PROPERTY_NAME_MPP_Y, 0))

    mpp_x_nm = mpp_x * 1000
    mpp_y_nm = mpp_y * 1000
    print(f"  Resolution: {mpp_x:.4f} µm/px  ({mpp_x_nm:.1f} nm/px)")

    offset_x_nm = float(props.get("hamamatsu.XOffsetFromSlideCentre", 0))
    offset_y_nm = float(props.get("hamamatsu.YOffsetFromSlideCentre", 0))

    slide.close()

    print(f"BBox count: {len(BOUNDING_BOXES)}")
    fragments = build_ndpa(
        BOUNDING_BOXES, img_w, img_h, mpp_x_nm, mpp_y_nm, offset_x_nm, offset_y_nm
    )

    out_path = NDPI_PATH + ".ndpa"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(pretty_join(fragments))

In [ ]:
df = pd.read_parquet("./results/microglia_features_extended_split.parquet")
vals = df["slide"].unique()
NDPI_DIR = "/mnt/d/microglia_data/"
ndpi_pathes = [f for f in Path(NDPI_DIR).glob("./*.ndpi")]
print(vals)
print(ndpi_pathes)

In [ ]:
slice = 5
for i, path in enumerate(ndpi_pathes):
    write_ndpa(df, str(path), slice, vals[i])

## Calculate Cell counts in freehand (slice) region

In [ ]:
import xml.etree.ElementTree as ET
from shapely.geometry import Polygon, Point


def count_annot_polygon(tree, area):
    root = tree.getroot()

    freehand_region = None
    rectangles = []

    for state in root.findall("ndpviewstate"):
        annotation = state.find("annotation")
        if annotation is None:
            continue

        points = [
            (int(p.find("x").text), int(p.find("y").text))
            for p in annotation.findall(".//point")
        ]

        if len(points) == 4:
            rectangles.append(
                {
                    "id": state.get("id"),
                    "points": points,
                }
            )
        elif len(points) > 4:
            freehand_region = {
                "id": state.get("id"),
                "polygon": Polygon(points),
            }

    if freehand_region is None:
        print("No freehand region found.")
    else:
        freehand_poly = freehand_region["polygon"]
        enclosed = []

        for rect in rectangles:
            rect_poly = Polygon(rect["points"])
            if freehand_poly.contains(rect_poly):
                enclosed.append(rect)
        print(f"Cells fully enclosed in slice: {len(enclosed)}")
        print(f"Cell density (mm^2): {len(enclosed)/area}")

In [ ]:
from pathlib import Path

ALL_DIRS = []
for slice_no in range(1, 6):
    ANNOT_DIR = f"/mnt/d/microglia_data/annotations/slice{slice_no}/"
    annotations = [f for f in sorted(Path(ANNOT_DIR).glob("./*.ndpa"))]
    ALL_DIRS.append(annotations)

areas_slices_1 = [18.7, 17.8, 9.76, 14.8, 16.1, 15.6, 16, 18.5, 24.9, 16.6, 16.6, 17.3]
areas_slices_2 = [26.2, 23.1, 22, 21.8, 21.6, 25.1, 25.1, 23.9, 18.9, 23.7, 24.5, 23.9]
areas_slices_3 = [29.8, 30.4, 26.4, 27, 28.4, 28.2, 27.1, 30.2, 23.5, 27.2, 28.9, 27.5]
areas_slices_4 = [
    25.5,
    30.3,
    28.8,
    28.2,
    30.6,
    27.2,
    29.6,
    32.3,
    33.4,
    29.2,
    30.5,
    28.1,
]
areas_slices_5 = [31.5, 29, 27.8, 27.2, 29.8, 28.6, 27.8, -1, 28.6, 25.3, -1, 30.9]

all_areas = [
    areas_slices_1,
    areas_slices_2,
    areas_slices_3,
    areas_slices_4,
    areas_slices_5,
]
print(len(ALL_DIRS))

## Calculate cell density per brain slice using area

In [ ]:
for slice_no, annotations in enumerate(ALL_DIRS):
    for idx, annot in enumerate(annotations):
        print(f"Slide: {annot.stem}, Slice: {slice_no+1}")
        tree = ET.parse(str(annot))
        area = areas_slices_1[idx]
        count_annot_polygon(tree, area)

## Mann-Whitney Test for significant difference in cell density

In [69]:
from scipy.stats import mannwhitneyu

ev_densities = [
    107,
    99,
    84,
    76,
    71.5,
    100,
    101,
    140,
    221,
    115,
    117,
    118,
    138,
    142,
    169.1,
    232.9,
    115.1,
    143.3,
    130.3,
    146,
    129.6,
    140.7,
    205.5,
    112.2,
    95.7,
    110.7,
    152,
    138.9,
    134.9,
    203,
    102.3,
    125.3,
    101.3,
    115.7,
]
tpo_densities = [
    112.9,
    148.3,
    67,
    113.9,
    88,
    168,
    120.3,
    88,
    106.3,
    112.4,
    169.63,
    115.4,
    69.3,
    121.3,
    103.7,
    133,
    108,
    58,
    101,
    74,
    96,
    78,
    75,
    81,
]


stats, p = mannwhitneyu(ev_densities, tpo_densities, alternative="two-sided")
print(stats)
print(f"P-value: {p}")

574.5
P-value: 0.008771267680360515


## Regional Density

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import random
import pyvips
import numpy as np
import cv2
from tqdm import tqdm
import pandas as pd
from scripts.filters import *
from scripts.utils import *
from cell_analysis.extract_features import *
import pandas as pd
from shapely.geometry import box as shapely_box


import xml.etree.ElementTree as ET
import numpy as np
from shapely.geometry import Polygon, box
import pandas as pd
import openslide
from scipy.stats import mannwhitneyu


def ndpa_nm_to_pixel(x_nm, y_nm, offset_x, offset_y, mpp_x, mpp_y, width, height):
    px = (x_nm - offset_x) / mpp_x + width / 2
    py = (y_nm - offset_y) / mpp_y + height / 2
    return px, py


def parse_ndpa_regions(ndpa_paths, ndpi_paths, slide_names):
    """
    Returns one row per freehand region with computed pixel area and mm^2 area.
    """
    regions = []

    for ndpa_path, ndpi_path, slide_name in zip(ndpa_paths, ndpi_paths, slide_names):
        slide = openslide.OpenSlide(str(ndpi_path))
        props = slide.properties

        offset_x = float(props.get("hamamatsu.XOffsetFromSlideCentre", 0))
        offset_y = float(props.get("hamamatsu.YOffsetFromSlideCentre", 0))

        mpp_x_nm = float(props.get(openslide.PROPERTY_NAME_MPP_X, 1)) * 1000
        mpp_y_nm = float(props.get(openslide.PROPERTY_NAME_MPP_Y, 1)) * 1000

        width, height = slide.dimensions

        tree = ET.parse(str(ndpa_path))
        root = tree.getroot()

        for state in root.findall(".//ndpviewstate"):
            title = state.findtext("title", default="").strip()
            annotation = state.find("annotation")
            if annotation is None:
                continue

            ann_type = annotation.get("type", "")
            if ann_type != "freehand":
                continue

            pointlist = annotation.find("pointlist")
            if pointlist is None:
                continue

            points_nm = []
            for pt in pointlist.findall("point"):
                x_txt = pt.findtext("x")
                y_txt = pt.findtext("y")
                if x_txt is None or y_txt is None:
                    continue
                points_nm.append((float(x_txt), float(y_txt)))

            points_px = [
                ndpa_nm_to_pixel(
                    x, y, offset_x, offset_y, mpp_x_nm, mpp_y_nm, width, height
                )
                for x, y in points_nm
            ]

            poly_px = Polygon(points_px)

            area_px2 = poly_px.area
            area_mm2 = area_px2 * mpp_x_nm * mpp_y_nm / 1e12

            regions.append(
                {
                    "slide": slide_name,
                    "region_id": state.get("id"),
                    "title": title,
                    "polygon_px": poly_px,
                    "area_px2": area_px2,
                    "area_mm2": area_mm2,
                }
            )

        slide.close()

    return regions


def summarize_cells_by_region(cells_df, regions):
    cells_df = cells_df.copy()
    cells_df["region_id"] = None
    cells_df["region_title"] = None
    cells_df["region_key"] = None

    regions_by_slide = {}
    for reg in regions:
        regions_by_slide.setdefault(reg["slide"], []).append(reg)

    region_counts = {(reg["slide"], reg["region_id"]): 0 for reg in regions}

    for idx, row in tqdm(cells_df.iterrows(), total=len(cells_df)):
        slide_name = row["slide"]
        if slide_name not in regions_by_slide:
            continue

        cell_poly = shapely_box(row["x1"], row["y1"], row["x2"], row["y2"])

        candidates = []
        for reg in regions_by_slide[slide_name]:
            poly = reg["polygon_px"]
            if poly.covers(cell_poly):
                candidates.append(reg)

        if not candidates:
            continue

        chosen = min(candidates, key=lambda r: r["area_px2"])
        key = (chosen["slide"], chosen["region_id"])

        cells_df.at[idx, "region_id"] = chosen["region_id"]
        cells_df.at[idx, "region_title"] = chosen["title"]
        cells_df.at[idx, "region_key"] = f"{chosen['slide']}::{chosen['region_id']}"
        region_counts[key] += 1

    summary_rows = []
    for reg in regions:
        key = (reg["slide"], reg["region_id"])
        n_cells = region_counts.get(key, 0)
        density = np.nan if reg["area_mm2"] <= 0 else n_cells / reg["area_mm2"]

        summary_rows.append(
            {
                "slide": reg["slide"],
                "region_id": reg["region_id"],
                "region_key": f"{reg['slide']}::{reg['region_id']}",
                "region_title": reg["title"],
                "area_px2": reg["area_px2"],
                "area_mm2": reg["area_mm2"],
                "cell_count": n_cells,
                "density_cells_per_mm2": density,
            }
        )

    region_summary_df = pd.DataFrame(summary_rows)
    return region_summary_df, cells_df


res = pd.read_parquet("./results/new2/microglia_feats_extended.parquet")
pd_regs = res

vals = res["slide"].unique()
NDPI_DIR = "/mnt/d/microglia_data/"
ndpi_pathes = sorted([f for f in Path(NDPI_DIR).glob("./*.ndpi")])
NDPA_DIR = "/mnt/d/microglia_data/annotations/brain_parts"
ndpa_pathes = sorted([f for f in Path(NDPA_DIR).glob("./*.ndpa")])

regions = parse_ndpa_regions(ndpa_pathes, ndpi_pathes, vals)
print(f"{len(regions)} regions read")
regions

218 regions read


[{'slide': 'TPO_60',
  'region_id': '1',
  'title': 'CTX',
  'polygon_px': <POLYGON ((42578.135 7131.873, 39987.607 7897.176, 39487.264 10899.503, 4019...>,
  'area_px2': 236663496.31052497,
  'area_mm2': 12.261109890445},
 {'slide': 'TPO_60',
  'region_id': '2',
  'title': 'HPF',
  'polygon_px': <POLYGON ((74299.74 14367.635, 75118.718 15520.209, 76240.951 16581.79, 7730...>,
  'area_px2': 32459431.152468394,
  'area_mm2': 1.6816647203569997},
 {'slide': 'TPO_60',
  'region_id': '3',
  'title': 'TH',
  'polygon_px': <POLYGON ((80035.622 15497.306, 81393.996 16629.353, 82865.675 17761.405, 83...>,
  'area_px2': 92283013.17235017,
  'area_mm2': 4.781016857973499},
 {'slide': 'TPO_60',
  'region_id': '4',
  'title': 'HY',
  'polygon_px': <POLYGON ((78462.921 25549.757, 78428.692 27660.636, 77584.347 30672.896, 76...>,
  'area_px2': 58221779.61167385,
  'area_mm2': 3.0163656371379997},
 {'slide': 'TPO_60',
  'region_id': '5',
  'title': 'CTX',
  'polygon_px': <POLYGON ((79389.692 34040.95

In [3]:
region_summary_df, res_with_regions = summarize_cells_by_region(
    res,
    regions,
)

res_with_regions

100%|██████████| 113364/113364 [00:30<00:00, 3706.76it/s]


,slide,label,x1,y1,x2,y2,cx,cy,soma_area,soma_radius,...,sholl_240,sholl_250,brain_slice,sholl_depth,sholl_max,sholl_auc,sholl_decay,region_id,region_title,region_key
0,TPO_60,1,4111,12348,4391,12721,115,200,1423,23.125277,...,0.0,0.0,1,22.0,15.0,1300.0,-0.020383,None,None,None
1,TPO_60,1,3967,12920,4352,13286,173,185,2299,35.524072,...,0.0,0.0,1,24.0,18.0,2370.0,-0.015984,None,None,None
2,TPO_60,1,3990,13359,4225,13568,141,120,1686,25.590498,...,0.0,0.0,1,13.0,10.0,705.0,-0.047848,None,None,None
3,TPO_60,1,4153,13580,4352,13824,100,101,1547,25.148600,...,0.0,0.0,1,13.0,12.0,1000.0,-0.039177,None,None,None
4,TPO_60,1,4155,13896,4352,14289,89,197,1101,23.071877,...,0.0,0.0,1,18.0,14.0,1695.0,-0.077022,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113359,TPO_67_TPO,1,156719,18058,156773,18116,28,22,1198,19.781183,...,0.0,0.0,5,1.0,3.0,15.0,NaN,None,None,None
113360,TPO_67_TPO,1,156672,18944,156889,19439,129,313,2821,34.587057,...,7.0,0.0,5,25.0,16.0,2115.0,-0.004579,None,None,None
113361,TPO_67_TPO,1,156600,19472,156801,19712,97,120,3871,43.284562,...,0.0,0.0,5,10.0,12.0,735.0,-0.030197,None,None,None
113362,TPO_67_TPO,1,156479,19712,156580,19824,58,56,3113,33.956610,...,0.0,0.0,5,2.0,7.0,65.0,-0.084730,None,None,None


In [8]:
res_with_regions.dropna(subset=["region_id"], inplace=True)
print(len(res_with_regions))

91725


In [9]:
res_with_regions.to_parquet("./results/new2/regional_density_all_cells.parquet")

In [3]:
region_summary_df.to_parquet("./results/new2/regional_density.parquet")
print(region_summary_df["cell_count"].sum())

91725


In [6]:
print(region_summary_df.columns)

Index(['slide', 'region_id', 'region_key', 'region_title', 'area_px2',
       'area_mm2', 'cell_count', 'density_cells_per_mm2'],
      dtype='object')


In [4]:
regions = ["CTX", "TH", "HPF", "HY", "MB", "PAL", "STR"]
results = []


def remove_outliers_iqr(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return series[(series >= lower) & (series <= upper)]


for region in regions:
    reg_df = region_summary_df[region_summary_df["region_title"] == region]
    ev_vals = reg_df[reg_df["slide"].str.contains("EV", case=False, na=False)]
    tpo_vals = reg_df[~reg_df["slide"].str.contains("EV", case=False, na=False)]

    ev_densities = ev_vals["density_cells_per_mm2"]
    tpo_densities = tpo_vals["density_cells_per_mm2"]
    ev_clean = remove_outliers_iqr(ev_densities)
    tpo_clean = remove_outliers_iqr(tpo_densities)
    if len(ev_clean) < 2 or len(tpo_clean) < 2:
        print(f"skipped {region}")
        continue
    stats, p = mannwhitneyu(ev_densities, tpo_densities, alternative="two-sided")
    results.append(
        {
            "region": region,
            "n_ev": len(ev_clean),
            "n_tpo": len(tpo_clean),
            "mean_ev": ev_clean.mean(),
            "mean_tpo": tpo_clean.mean(),
            "median_ev": ev_clean.median(),
            "median_tpo": tpo_clean.median(),
            "stat": stats,
            "p_value": p,
        }
    )


results_df = pd.DataFrame(results).sort_values("p_value").reset_index(drop=True)
results_df.to_parquet("./results/new2/regional_density_stats.parquet")
results_df

,region,n_ev,n_tpo,mean_ev,mean_tpo,median_ev,median_tpo,stat,p_value
0,CTX,35,25,82.904118,73.985515,81.743458,72.519508,603.0,0.013360
1,HY,15,10,64.576841,67.614958,63.924106,68.385490,62.0,0.488074
2,MB,6,5,51.126201,51.261848,47.493511,47.374862,12.0,0.662338
3,HPF,17,13,76.212317,73.580974,75.805433,74.227904,130.0,0.676833
4,STR,27,19,78.395607,75.506323,78.807793,78.619163,275.0,0.688004
5,TH,14,10,58.963958,59.483328,60.294774,60.435640,70.0,0.721959
6,PAL,14,6,83.643431,82.525581,79.073268,74.242736,44.0,0.904386


In [5]:
print(results_df.to_latex(index=False))

\begin{tabular}{lrrrrrrrr}
\toprule
region & n_ev & n_tpo & mean_ev & mean_tpo & median_ev & median_tpo & stat & p_value \\
\midrule
CTX & 35 & 25 & 82.904118 & 73.985515 & 81.743458 & 72.519508 & 603.000000 & 0.013360 \\
HY & 15 & 10 & 64.576841 & 67.614958 & 63.924106 & 68.385490 & 62.000000 & 0.488074 \\
MB & 6 & 5 & 51.126201 & 51.261848 & 47.493511 & 47.374862 & 12.000000 & 0.662338 \\
HPF & 17 & 13 & 76.212317 & 73.580974 & 75.805433 & 74.227904 & 130.000000 & 0.676833 \\
STR & 27 & 19 & 78.395607 & 75.506323 & 78.807793 & 78.619163 & 275.000000 & 0.688004 \\
TH & 14 & 10 & 58.963958 & 59.483328 & 60.294774 & 60.435640 & 70.000000 & 0.721959 \\
PAL & 14 & 6 & 83.643431 & 82.525581 & 79.073268 & 74.242736 & 44.000000 & 0.904386 \\
\bottomrule
\end{tabular}

